In [1]:
# ============================================================
# CELL 1 — Imports & Configuration
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
CPCB_DIR = ROOT / "data" / "raw" / "cpcb"
OUTPUT_FILE = ROOT / "data" / "intermediate" / "cpcb_master_2021_2025.csv"

EXPECTED_STATIONS = 37
YEARS = list(range(2021, 2026))

POLLUTANT_MAP = {
    "PM2.5 (µg/m³)": "pm25",
    "PM10 (µg/m³)": "pm10",
    "NO2 (µg/m³)": "no2",
    "SO2 (µg/m³)": "so2",
    "CO (mg/m³)": "co",
    "Ozone (µg/m³)": "o3"
}

print("Configuration loaded.")
print("CPCB input :", CPCB_DIR)
print("Output     :", OUTPUT_FILE)
print("Stations   :", EXPECTED_STATIONS)
print("Years      :", YEARS)

Configuration loaded.
CPCB input : C:\Users\udish\College\MajorProjectSem7\Implementation\data\raw\cpcb
Output     : C:\Users\udish\College\MajorProjectSem7\Implementation\data\intermediate\cpcb_master_2021_2025.csv
Stations   : 37
Years      : [2021, 2022, 2023, 2024, 2025]


In [2]:
# ============================================================
# CELL 2 — Station Registry + Coordinates
# ============================================================

stations = {
    "ST01":{"name":"Alipur","city":"Delhi","state":"Delhi","latitude":28.8153,"longitude":77.1530},
    "ST02":{"name":"Anand Vihar Delhi","city":"Delhi","state":"Delhi","latitude":28.6476,"longitude":77.3158},
    "ST03":{"name":"Anand Vihar Hapur","city":"Hapur","state":"Uttar Pradesh","latitude":28.7256,"longitude":77.7497},
    "ST04":{"name":"Arya Nagar Bahadurgarh","city":"Bahadurgarh","state":"Haryana","latitude":28.6701,"longitude":76.9254},
    "ST05":{"name":"Ashok Vihar","city":"Delhi","state":"Delhi","latitude":28.6954,"longitude":77.1817},
    "ST06":{"name":"Aya Nagar","city":"Delhi","state":"Delhi","latitude":28.4707,"longitude":77.1099},
    "ST07":{"name":"Bawana","city":"Delhi","state":"Delhi","latitude":28.7762,"longitude":77.0511},
    "ST08":{"name":"Chandni Chowk","city":"Delhi","state":"Delhi","latitude":28.6568,"longitude":77.2272},
    "ST09":{"name":"IGI Airport T3","city":"Delhi","state":"Delhi","latitude":28.5628,"longitude":77.1180},
    "ST10":{"name":"ITO","city":"Delhi","state":"Delhi","latitude":28.6286,"longitude":77.2411},
    "ST11":{"name":"Jahangirpuri","city":"Delhi","state":"Delhi","latitude":28.7328,"longitude":77.1706},
    "ST12":{"name":"Jai Bhim Nagar Meerut","city":"Meerut","state":"Uttar Pradesh","latitude":28.9536,"longitude":77.7623},
    "ST13":{"name":"Knowledge Park III Greater Noida","city":"Greater Noida","state":"Uttar Pradesh","latitude":28.4727,"longitude":77.4820},
    "ST14":{"name":"Lodhi Road IMD","city":"Delhi","state":"Delhi","latitude":28.5918,"longitude":77.2273},
    "ST15":{"name":"Loni Ghaziabad","city":"Ghaziabad","state":"Uttar Pradesh","latitude":28.7573,"longitude":77.2788},
    "ST16":{"name":"Mandir Marg","city":"Delhi","state":"Delhi","latitude":28.6364,"longitude":77.2011},
    "ST17":{"name":"Municipal Corporation Office Dharuhera","city":"Dharuhera","state":"Haryana","latitude":28.2068,"longitude":76.7997},
    "ST18":{"name":"Mundka","city":"Delhi","state":"Delhi","latitude":28.6847,"longitude":77.0766},
    "ST19":{"name":"Najafgarh","city":"Delhi","state":"Delhi","latitude":28.5702,"longitude":76.9338},
    "ST20":{"name":"Narela","city":"Delhi","state":"Delhi","latitude":28.8228,"longitude":77.1020},
    "ST21":{"name":"New Industrial Township Faridabad","city":"Faridabad","state":"Haryana","latitude":28.3907,"longitude":77.3006},
    "ST22":{"name":"NSUT Dwarka","city":"Delhi","state":"Delhi","latitude":28.6091,"longitude":77.0325},
    "ST23":{"name":"Okhla Phase 2","city":"Delhi","state":"Delhi","latitude":28.5308,"longitude":77.2713},
    "ST24":{"name":"Punjabi Bagh","city":"Delhi","state":"Delhi","latitude":28.6740,"longitude":77.1310},
    "ST25":{"name":"R K Puram","city":"Delhi","state":"Delhi","latitude":28.5633,"longitude":77.1869},
    "ST26":{"name":"Rohini","city":"Delhi","state":"Delhi","latitude":28.7325,"longitude":77.1199},
    "ST27":{"name":"Sanjay Nagar Ghaziabad","city":"Ghaziabad","state":"Uttar Pradesh","latitude":28.6854,"longitude":77.4538},
    "ST28":{"name":"Sardar Patel Inter College Baghpat","city":"Baghpat","state":"Uttar Pradesh","latitude":28.9649,"longitude":77.2788},
    "ST29":{"name":"Sector 2 IMT Manesar","city":"Manesar","state":"Haryana","latitude":28.3607,"longitude":76.9361},
    "ST30":{"name":"Sector 11 Faridabad","city":"Faridabad","state":"Haryana","latitude":28.3761,"longitude":77.3157},
    "ST31":{"name":"Sector 116 Noida","city":"Noida","state":"Uttar Pradesh","latitude":28.5692,"longitude":77.3938},
    "ST32":{"name":"Sector 51 Gurugram","city":"Gurugram","state":"Haryana","latitude":28.4227,"longitude":77.0667},
    "ST33":{"name":"Sector 62 Noida","city":"Noida","state":"Uttar Pradesh","latitude":28.6245,"longitude":77.3577},
    "ST34":{"name":"Shyam Nagar Palwal","city":"Palwal","state":"Haryana","latitude":28.1486,"longitude":77.3321},
    "ST35":{"name":"Sonia Vihar","city":"Delhi","state":"Delhi","latitude":28.7105,"longitude":77.2495},
    "ST36":{"name":"Vasundhara Ghaziabad","city":"Ghaziabad","state":"Uttar Pradesh","latitude":28.6603,"longitude":77.3573},
    "ST37":{"name":"Vikas Sadan Gurugram","city":"Gurugram","state":"Haryana","latitude":28.4501,"longitude":77.0263}
}

station_df = pd.DataFrame(
    [{"station_code":k, **v} for k,v in stations.items()]
)

print("Stations registered:", len(station_df))
print(
    "Missing coordinates:",
    station_df[["latitude","longitude"]].isna().any(axis=1).sum()
)

Stations registered: 37
Missing coordinates: 0


In [3]:
# ============================================================
# CELL 3 — Locate CPCB Station Files
# ============================================================

station_dirs = sorted(
    p for p in CPCB_DIR.iterdir()
    if p.is_dir() and p.name.startswith("ST")
)

print("="*60)
print("CPCB FILE INVENTORY")
print("="*60)
print("Station folders found:", len(station_dirs))
print("Expected stations     :", EXPECTED_STATIONS)

if len(station_dirs) != EXPECTED_STATIONS:
    raise ValueError(
        f"Expected {EXPECTED_STATIONS} station folders, "
        f"found {len(station_dirs)}."
    )

for station_dir in station_dirs:
    files = sorted(station_dir.glob("file_*.csv"))
    print(f"{station_dir.name}: {len(files)} files")

CPCB FILE INVENTORY
Station folders found: 37
Expected stations     : 37
ST01: 5 files
ST02: 5 files
ST03: 5 files
ST04: 5 files
ST05: 5 files
ST06: 5 files
ST07: 5 files
ST08: 5 files
ST09: 5 files
ST10: 5 files
ST11: 5 files
ST12: 5 files
ST13: 5 files
ST14: 5 files
ST15: 5 files
ST16: 5 files
ST17: 5 files
ST18: 5 files
ST19: 5 files
ST20: 5 files
ST21: 5 files
ST22: 5 files
ST23: 5 files
ST24: 5 files
ST25: 5 files
ST26: 5 files
ST27: 5 files
ST28: 5 files
ST29: 5 files
ST30: 5 files
ST31: 5 files
ST32: 5 files
ST33: 5 files
ST34: 5 files
ST35: 5 files
ST36: 5 files
ST37: 5 files


In [4]:
# ============================================================
# CELL 4 — Validate Station/Year File Coverage
# ============================================================

records = []

for station_dir in station_dirs:
    station_code = station_dir.name
    files = sorted(station_dir.glob("file_*.csv"))

    if len(files) != 5:
        raise ValueError(
            f"{station_code} has {len(files)} files; expected 5."
        )

    for idx, file_path in enumerate(files, start=2021):
        expected_name = f"file_{idx - 2020:02d}.csv"

        if file_path.name != expected_name:
            raise ValueError(
                f"{station_code}: expected {expected_name}, "
                f"found {file_path.name}"
            )

        records.append({
            "station_code": station_code,
            "year": idx,
            "path": file_path
        })

file_map = pd.DataFrame(records)

coverage = (
    file_map
    .assign(present=1)
    .pivot_table(
        index="station_code",
        columns="year",
        values="present",
        fill_value=0
    )
    .reindex(
        index=sorted(stations),
        columns=YEARS,
        fill_value=0
    )
)

print("="*60)
print("FILE COVERAGE")
print("="*60)
print(coverage.to_string())

if not (coverage == 1).all().all():
    raise ValueError("Incomplete station/year coverage.")

print("\nPASS — 37 stations × 5 years = 185 files.")

FILE COVERAGE
year          2021  2022  2023  2024  2025
station_code                              
ST01           1.0   1.0   1.0   1.0   1.0
ST02           1.0   1.0   1.0   1.0   1.0
ST03           1.0   1.0   1.0   1.0   1.0
ST04           1.0   1.0   1.0   1.0   1.0
ST05           1.0   1.0   1.0   1.0   1.0
ST06           1.0   1.0   1.0   1.0   1.0
ST07           1.0   1.0   1.0   1.0   1.0
ST08           1.0   1.0   1.0   1.0   1.0
ST09           1.0   1.0   1.0   1.0   1.0
ST10           1.0   1.0   1.0   1.0   1.0
ST11           1.0   1.0   1.0   1.0   1.0
ST12           1.0   1.0   1.0   1.0   1.0
ST13           1.0   1.0   1.0   1.0   1.0
ST14           1.0   1.0   1.0   1.0   1.0
ST15           1.0   1.0   1.0   1.0   1.0
ST16           1.0   1.0   1.0   1.0   1.0
ST17           1.0   1.0   1.0   1.0   1.0
ST18           1.0   1.0   1.0   1.0   1.0
ST19           1.0   1.0   1.0   1.0   1.0
ST20           1.0   1.0   1.0   1.0   1.0
ST21           1.0   1.0   1.0   1.0   1

In [5]:
# ============================================================
# CELL 5 — Load, Standardize & Merge CPCB Data
# ============================================================

meta = station_df.set_index("station_code")
frames = []

for _, row in file_map.iterrows():

    df = pd.read_csv(row["path"])

    required_cols = ["Timestamp"] + list(POLLUTANT_MAP.keys())
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"{row['path'].name} is missing columns: {missing_cols}"
        )

    df = df[
        ["Timestamp"] + list(POLLUTANT_MAP.keys())
    ].copy()

    df["station_code"] = row["station_code"]
    df["source_year"] = row["year"]

    frames.append(df)

master_df = pd.concat(frames, ignore_index=True)

master_df["datetime_utc"] = pd.to_datetime(
    master_df.pop("Timestamp"),
    errors="coerce"
)

master_df = (
    master_df
    .join(
        meta[
            ["name","city","state","latitude","longitude"]
        ],
        on="station_code"
    )
    .rename(
        columns={
            "name":"station_name",
            **POLLUTANT_MAP
        }
    )
)

master_df["location_id"] = master_df["station_code"]

for col in (
    ["source_year"]
    + list(POLLUTANT_MAP.values())
    + ["latitude","longitude"]
):
    master_df[col] = pd.to_numeric(
        master_df[col],
        errors="coerce"
    )

master_df = master_df[
    [
        "station_code",
        "location_id",
        "station_name",
        "city",
        "state",
        "latitude",
        "longitude",
        "source_year",
        "datetime_utc",
    ]
    + list(POLLUTANT_MAP.values())
].sort_values(
    ["datetime_utc","station_code"]
).reset_index(drop=True)

print("="*60)
print("CPCB MASTER DATASET CREATED")
print("="*60)
print("Rows     :", f"{len(master_df):,}")
print("Stations :", master_df.location_id.nunique())
print("Years    :", sorted(master_df.source_year.unique()))
print("Columns  :", len(master_df.columns))
print("Date     :", master_df.datetime_utc.min(), "→",
      master_df.datetime_utc.max())

CPCB MASTER DATASET CREATED
Rows     : 1,621,488
Stations : 37
Years    : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Columns  : 15
Date     : 2021-01-01 00:00:00 → 2025-12-31 23:00:00


In [ ]:
# ============================================================
# CELL 6 — Master Dataset Quality Audit
# ============================================================

pollutants = list(POLLUTANT_MAP.values())

expected_per_year = {
    y: 8784 if y == 2024 else 8760
    for y in YEARS
}

print("="*60)
print("CPCB MASTER DATA QUALITY AUDIT")
print("="*60)

print(
    f"Rows: {len(master_df):,} | "
    f"Stations: {master_df.location_id.nunique()} | "
    f"Columns: {len(master_df.columns)}"
)

print("Invalid timestamps:",master_df.datetime_utc.isna().sum())

print("Duplicate rows:",master_df.duplicated().sum())

print("Duplicate station-timestamps:",master_df.duplicated(["location_id","datetime_utc"]).sum())

print("Missing station IDs:",master_df.location_id.isna().sum())

print("\nRows by station:")
print(
    master_df
    .groupby(["location_id","station_name"])
    .size()
    .to_string()
)

print("\nRows by year:")
print(
    master_df
    .groupby("source_year")
    .size()
    .to_string()
)

print("\nMissing pollutant values:")
print(
    pd.DataFrame({
        "count":master_df[pollutants].isna().sum(),
        "%":(
            master_df[pollutants].isna().mean()*100
        ).round(2)
    })
)

print("\nInvalid pollutant values:")

for col in pollutants:
    s = master_df[col]

    print(
        f"{col:5} | "
        f"negative={(s < 0).sum():,} | "
        f"inf={np.isinf(s).sum():,}"
    )

print("\nTimestamp coverage by station-year:")

coverage_check = (
    master_df
    .groupby(["location_id","source_year"])["datetime_utc"]
    .agg(["min","max","nunique"])
    .reset_index()
)

coverage_check["expected"] = (
    coverage_check["source_year"]
    .map(expected_per_year)
)

coverage_check["coverage_%"] = (
    coverage_check["nunique"]
    / coverage_check["expected"]
    * 100
).round(2)

print(coverage_check.to_string(index=False))

CPCB MASTER DATA QUALITY AUDIT
Rows: 1,621,488 | Stations: 37 | Columns: 15
Invalid timestamps: 0
Duplicate rows: 0
Duplicate station-timestamps: 0
Missing station IDs: 0

Rows by station:
location_id  station_name                          
ST01         Alipur                                    43824
ST02         Anand Vihar Delhi                         43824
ST03         Anand Vihar Hapur                         43824
ST04         Arya Nagar Bahadurgarh                    43824
ST05         Ashok Vihar                               43824
ST06         Aya Nagar                                 43824
ST07         Bawana                                    43824
ST08         Chandni Chowk                             43824
ST09         IGI Airport T3                            43824
ST10         ITO                                       43824
ST11         Jahangirpuri                              43824
ST12         Jai Bhim Nagar Meerut                     43824
ST13         Knowledge Park

In [7]:
# ============================================================
# CELL 7 — Controlled CPCB Cleaning
# ============================================================

df_raw = master_df.copy()
df_clean = master_df.copy()

for col in pollutants:

    df_clean.loc[
        df_clean[col] < 0,
        col
    ] = np.nan

    df_clean.loc[
        np.isinf(df_clean[col]),
        col
    ] = np.nan

df_clean = df_clean.dropna(
    subset=["datetime_utc","location_id"]
)

df_clean = df_clean.drop_duplicates()

df_clean = df_clean.drop_duplicates(
    ["location_id","datetime_utc"],
    keep="first"
)

df_clean = df_clean.sort_values(
    ["datetime_utc","location_id"]
).reset_index(drop=True)

print("="*60)
print("CONTROLLED CPCB CLEANING")
print("="*60)

print("Original rows :", f"{len(df_raw):,}")
print("Cleaned rows  :", f"{len(df_clean):,}")
print("Rows removed  :", f"{len(df_raw)-len(df_clean):,}")

print(
    "Negative values remaining:",
    (df_clean[pollutants] < 0).sum().sum()
)

print(
    "Infinite values remaining:",
    np.isinf(df_clean[pollutants]).sum().sum()
)

print("\nMissing values after cleaning:")

print(
    pd.DataFrame({
        "count":df_clean[pollutants].isna().sum(),
        "%":(
            df_clean[pollutants].isna().mean()*100
        ).round(2)
    })
)

CONTROLLED CPCB CLEANING
Original rows : 1,621,488
Cleaned rows  : 1,621,488
Rows removed  : 0
Negative values remaining: 0
Infinite values remaining: 0

Missing values after cleaning:
       count      %
pm25  183852  11.34
pm10  212179  13.09
no2   175889  10.85
so2   355884  21.95
co    216539  13.35
o3    185043  11.41


In [8]:
# ============================================================
# CELL 8 — Save Final CPCB Master Dataset
# ============================================================

FINAL_COLUMNS = [
    "location_id",
    "station_name",
    "city",
    "state",
    "latitude",
    "longitude",
    "source_year",
    "datetime_utc",
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3"
]

df_clean = df_clean[FINAL_COLUMNS]

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_clean.to_csv(
    OUTPUT_FILE,
    index=False
)

print("="*60)
print("CPCB MASTER DATASET SAVED")
print("="*60)

print("Rows       :", f"{len(df_clean):,}")
print("Stations   :", df_clean.location_id.nunique())
print("Years      :", sorted(df_clean.source_year.unique()))
print(
    "Date range :",
    df_clean.datetime_utc.min(),
    "→",
    df_clean.datetime_utc.max()
)

print(
    "Missing coordinates:",
    df_clean[
        ["latitude","longitude"]
    ].isna().any(axis=1).sum()
)

print("Columns    :", list(df_clean.columns))
print("File       :", OUTPUT_FILE)
print("="*60)

CPCB MASTER DATASET SAVED
Rows       : 1,621,488
Stations   : 37
Years      : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Date range : 2021-01-01 00:00:00 → 2025-12-31 23:00:00
Missing coordinates: 0
Columns    : ['location_id', 'station_name', 'city', 'state', 'latitude', 'longitude', 'source_year', 'datetime_utc', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3']
File       : C:\Users\udish\College\MajorProjectSem7\Implementation\data\intermediate\cpcb_master_2021_2025.csv
